# 09 — Pandas: The Backbone of ML Data Pipelines

| | |
|---|---|
| **Difficulty** | ⭐⭐⭐ (Intermediate) |
| **Estimated Time** | 3.5 hours |
| **Week** | Week 3 — Statistics for ML & Python Data Stack |
| **Prerequisites** | Python basics, NumPy arrays |

---

## What You Will Learn
By the end of this notebook you will be able to:
- Create and explore DataFrames from multiple sources
- Index, slice, and filter data with `.loc`, `.iloc`, and boolean conditions
- Handle missing values with principled strategies
- Engineer new features using column arithmetic and `apply`
- Aggregate and summarise data with `groupby`
- Combine datasets using `merge`

## 1. Why Does This Matter for Machine Learning?

Before any ML algorithm sees data, **a human (or a pipeline) must clean and shape it**. In practice, data scientists spend 60–80 % of their time on data wrangling, not modelling. Pandas is the standard tool for that wrangling step in Python.

Here is how pandas fits into the ML workflow:

```
Raw CSV / Database
       |
    pandas          ← YOU ARE HERE
  (clean, explore, engineer features)
       |
  NumPy array  ←  sklearn / PyTorch / TensorFlow
       |
  ML Model
```

Every scikit-learn tutorial starts with `pd.read_csv(...)`. Every Kaggle competition winner publishes a pandas-heavy EDA notebook. **Knowing pandas deeply is a force-multiplier for all downstream ML work.**

## 2. Real-World Analogy — Excel on Steroids

Imagine you manage a real-estate agency and keep a giant **Excel spreadsheet** of every house you have ever listed:

| house_id | neighborhood | size_sqft | bedrooms | price |
|----------|-------------|-----------|----------|-------|
| H001 | Greenwood | 1 850 | 3 | 320 000 |
| H002 | Downtown | 950 | 1 | 210 000 |
| … | … | … | … | … |

**Pandas is that spreadsheet — but with superpowers:**

| Excel Action | Pandas Equivalent |
|---|---|
| Filter rows (AutoFilter) | Boolean indexing |
| Pivot Table | `groupby().agg()` |
| VLOOKUP | `pd.merge()` |
| Fill formula down a column | Vectorised column arithmetic |
| Macro / formula | `apply()` with a Python function |

The key difference: pandas operates on **millions of rows in milliseconds**, works inside Python scripts and Jupyter notebooks, integrates seamlessly with ML libraries, and never crashes because someone accidentally edited a cell.

## 3. Core Concepts

### Series — a single labelled column
A **Series** is a one-dimensional array with an **index** (row labels). Think of it as a Python dictionary merged with a NumPy array: you get both fast numeric operations AND label-based lookup.

```
Index   Value
  0     320 000   ← house H001
  1     210 000   ← house H002
  2     415 000   ← house H003
Name: price
```

### DataFrame — a collection of Series sharing the same index
A **DataFrame** is a 2-D table where every column is a Series. All columns share the same row index, which is why you can align columns effortlessly and do row-wise operations.

```
         size_sqft   bedrooms   price
Index
  0          1850        3      320 000
  1           950        1      210 000
  2          2200        4      415 000
```

### Why this design matters for ML
- Each **row** = one training example (one house)
- Each **column** = one feature (size, bedrooms, …)
- The **target column** (price) lives alongside the features until you explicitly separate them with `X = df.drop('price', axis=1)` and `y = df['price']`

## 4. Setup — Imports and Synthetic Dataset

In [ ]:
# ── Standard imports ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Suppress minor warnings so output stays readable
warnings.filterwarnings('ignore')

# Make pandas display wider DataFrames without wrapping
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:,.2f}'.format)  # comma-thousands in floats

# Set a random seed so results are reproducible every time you run this notebook
np.random.seed(42)

print('Pandas version:', pd.__version__)
print('NumPy version:', np.__version__)

In [ ]:
# ── Build a synthetic house-price dataset with 200 rows ──────────────────────
# We mimic the kind of messy, real-world data a data scientist actually receives.

N = 200  # number of houses

# Four neighbourhoods with distinct price levels (reflects location premium)
neighborhoods = np.random.choice(
    ['Greenwood', 'Downtown', 'Riverside', 'Suburbia'],
    size=N,
    p=[0.30, 0.20, 0.25, 0.25]   # unequal neighbourhood distribution
)

# Square footage: normally distributed around 1800 sq ft, clipped to realistic range
size_sqft = np.random.normal(loc=1800, scale=500, size=N).clip(600, 4500).astype(int)

# Number of bedrooms: 1–6, skewed toward 3
bedrooms = np.random.choice([1, 2, 3, 4, 5, 6], size=N, p=[0.05, 0.20, 0.40, 0.25, 0.07, 0.03])

# Bathrooms: correlated with bedrooms (more beds → more baths)
bathrooms = (bedrooms * np.random.uniform(0.5, 0.9, size=N)).clip(1, 5).astype(int)

# Age of the house in years
age_years = np.random.randint(0, 60, size=N)

# Garage: 1 = has garage, 0 = no garage
garage = np.random.choice([0, 1], size=N, p=[0.25, 0.75])

# ── Price model ─────────────────────────────────────────────────────────────
# Base price depends on size, bedrooms, age, and neighbourhood premium
neighborhood_premium = {'Greenwood': 50_000, 'Downtown': 80_000,
                        'Riverside': 30_000, 'Suburbia': 10_000}
premiums = np.array([neighborhood_premium[n] for n in neighborhoods])

price = (
    100 * size_sqft                        # $100 per square foot base
    + 15_000 * bedrooms                    # premium per bedroom
    - 1_000 * age_years                    # older houses worth less
    + 20_000 * garage                      # garage premium
    + premiums                             # neighbourhood premium
    + np.random.normal(0, 20_000, size=N)  # random market noise
).clip(80_000, 1_200_000).astype(int)

# Sold date: random dates in 2023
start = pd.Timestamp('2023-01-01')
sold_date = [start + pd.Timedelta(days=int(d)) for d in np.random.randint(0, 365, N)]

# ── Assemble the DataFrame ───────────────────────────────────────────────────
df = pd.DataFrame({
    'house_id':     [f'H{i:03d}' for i in range(1, N + 1)],
    'neighborhood': neighborhoods,
    'size_sqft':    size_sqft,
    'bedrooms':     bedrooms,
    'bathrooms':    bathrooms,
    'age_years':    age_years,
    'garage':       garage,
    'price':        price,
    'sold_date':    sold_date,
})

print(f'Dataset shape: {df.shape}')   # (200, 9)
df.head()

## 5. Exploring a DataFrame

When you first receive a dataset, your instinct should be to **inspect before you manipulate**. These methods give you a rapid health-check.

In [ ]:
# ── Shape and dimensions ─────────────────────────────────────────────────────
print('Shape (rows, cols):', df.shape)           # (200, 9)
print('Number of rows    :', len(df))            # equivalent to df.shape[0]
print('Number of columns :', df.shape[1])
print()

# ── Data types ───────────────────────────────────────────────────────────────
# dtypes tells you whether pandas stored a column as int, float, object (string), etc.
# Wrong dtypes (e.g., price stored as string) cause silent arithmetic errors in ML
print('Column data types:')
print(df.dtypes)
print()

# ── Concise summary ──────────────────────────────────────────────────────────
# .info() combines dtype info WITH non-null counts — great for spotting missing data
print('DataFrame info:')
df.info()

In [ ]:
# ── Statistical summary ──────────────────────────────────────────────────────
# .describe() computes count, mean, std, min, quartiles, max for every numeric column
# This is your first sanity check: are values in a plausible range?
df.describe()

# Key things to look for:
#   - min/max: are there impossible values (e.g., negative price)?
#   - mean vs 50th percentile: large gap → skewed distribution
#   - std relative to mean: high coefficient of variation → high spread

In [ ]:
# ── First and last rows ──────────────────────────────────────────────────────
print('=== First 5 rows (head) ===')
print(df.head())         # shows the first 5 rows by default; df.head(10) for 10
print()

print('=== Last 5 rows (tail) ===')
print(df.tail())         # useful for confirming the dataset ends correctly
print()

# ── Missing value audit ──────────────────────────────────────────────────────
# isnull() returns a boolean DataFrame; .sum() counts Trues per column
# Right now our dataset is complete — we'll inject NaNs in Section 7
print('=== Missing values per column ===')
print(df.isnull().sum())
print()

# ── Unique value counts ──────────────────────────────────────────────────────
# For categorical columns, value_counts() shows frequency distribution
print('=== Neighbourhood counts ===')
print(df['neighborhood'].value_counts())

## 6. Indexing — Selecting Rows and Columns

Pandas provides **three** indexing styles. Mixing them up is the #1 source of bugs for beginners.

| Method | Based on | Syntax |
|--------|----------|--------|
| `[]` | Column name (simple) | `df['price']` |
| `.loc[]` | **Labels** (row label, column name) | `df.loc[0, 'price']` |
| `.iloc[]` | **Integer positions** (0-based) | `df.iloc[0, 7]` |

**Rule of thumb:** Prefer `.loc` when you know the label; prefer `.iloc` when you know the position (e.g., "give me the last column").

In [ ]:
# ── .loc — label-based indexing ──────────────────────────────────────────────
# df.loc[row_label, column_label]
# With the default integer index, row label == row position, BUT this is a coincidence.
# If you sort or filter the DataFrame the row labels change!

print('Single cell via .loc:')
print(df.loc[0, 'price'])   # price of the first house
print()

print('First 5 rows, three columns via .loc:')
print(df.loc[0:4, ['house_id', 'neighborhood', 'price']])  # NOTE: .loc slicing is INCLUSIVE
print()

# ── .iloc — position-based indexing ─────────────────────────────────────────
# df.iloc[row_position, column_position]
# Uses Python-style 0-based indexing; slices are EXCLUSIVE on the right

print('Single cell via .iloc (row 0, column 7 = price):')
print(df.iloc[0, 7])        # same cell as above
print()

print('Rows 0-4, columns 0-3 via .iloc:')
print(df.iloc[0:5, 0:4])    # NOTE: .iloc slicing is EXCLUSIVE (5 means up to but not including 5)
print()

print('Last 3 rows, last 2 columns via .iloc (negative indexing):')
print(df.iloc[-3:, -2:])

In [ ]:
# ── Boolean indexing — filter rows by condition ───────────────────────────────
# A boolean condition on a column produces a Series of True/False values.
# Passing that Series into df[...] keeps only the True rows.

# Single condition: houses priced above $400 000
expensive = df[df['price'] > 400_000]
print(f'Houses priced above $400K: {len(expensive)}')
print(expensive[['house_id', 'neighborhood', 'price']].head())
print()

# Multiple conditions: MUST use & (and) | (or) NOT Python keywords and/or
# ALWAYS wrap each condition in parentheses!
affordable_large = df[
    (df['price'] < 400_000) &
    (df['bedrooms'] >= 3)   &
    (df['garage'] == 1)
]
print(f'Affordable (< $400K), 3+ bedrooms, with garage: {len(affordable_large)}')
print(affordable_large[['house_id', 'neighborhood', 'bedrooms', 'price']].head())
print()

# isin() — check membership in a list (like SQL IN clause)
prime_locations = df[df['neighborhood'].isin(['Greenwood', 'Downtown'])]
print(f'Prime location houses: {len(prime_locations)}')

# ~ (tilde) = logical NOT
not_prime = df[~df['neighborhood'].isin(['Greenwood', 'Downtown'])]
print(f'Non-prime location houses: {len(not_prime)}')

## 7. Adding, Modifying, and Dropping Columns

In [ ]:
# ── Derived features — core feature engineering step ─────────────────────────
# In ML, raw features are rarely the best representation.
# Price per sq ft normalises for size and is often more predictive than raw price.

# Vectorised arithmetic: operates on every row simultaneously (no loops needed)
df['price_per_sqft'] = df['price'] / df['size_sqft']

# Log-transform: prices are right-skewed (a few very expensive houses).
# Log makes the distribution more symmetric → linear models perform better.
df['log_price'] = np.log(df['price'])   # natural log

# Age category: bin continuous age into interpretable groups
# pd.cut() divides a range into equal-width bins with custom labels
df['age_category'] = pd.cut(
    df['age_years'],
    bins=[0, 10, 30, 60],           # edges of bins
    labels=['new', 'mid', 'old'],   # label for each bin
    include_lowest=True             # include the left edge of the first bin
)

# ── Rename a column (often needed after merges or when names are cryptic) ─────
df = df.rename(columns={'size_sqft': 'sqft'})  # shorter name for convenience

# ── Drop a column you no longer need ─────────────────────────────────────────
# axis=1 means column axis; inplace=False (default) returns a new DataFrame
df_no_log = df.drop(columns=['log_price'])  # we'll keep log_price in main df

print('Columns after feature engineering:', df.columns.tolist())
df[['house_id', 'sqft', 'price', 'price_per_sqft', 'log_price', 'age_category']].head(6)

## 8. Handling Missing Values

### Plain English first
Missing values (`NaN`) are like blank cells in a spreadsheet. Most ML algorithms **cannot handle NaN** — they will throw an error or silently produce garbage predictions. You must decide:

1. **Drop the row** — safe if few rows are missing AND the missingness is random
2. **Fill with a statistic** — use mean/median/mode to preserve dataset size
3. **Fill with a domain-informed value** — e.g., fill missing garage flag with 0 (assume no garage)
4. **Forward/backward fill** — for time-series data where the previous value is a good proxy

**Decision rule of thumb:**
- < 5 % missing → safe to drop or fill with mean
- 5–20 % missing → fill carefully (consider using median, mode, or a model)
- > 20 % missing → the column may not be reliable at all; consider dropping the column

In [ ]:
# ── Inject realistic missing values to practise on ───────────────────────────
df_missing = df.copy()   # always work on a copy; never mutate the original

# Randomly set 15 price values to NaN  (7.5% missing)
missing_price_idx = np.random.choice(df_missing.index, size=15, replace=False)
df_missing.loc[missing_price_idx, 'price'] = np.nan

# Randomly set 10 sqft values to NaN  (5% missing)
missing_sqft_idx = np.random.choice(df_missing.index, size=10, replace=False)
df_missing.loc[missing_sqft_idx, 'sqft'] = np.nan

# ── Audit missing values ─────────────────────────────────────────────────────
print('Missing value counts after injection:')
missing_counts = df_missing.isnull().sum()
print(missing_counts[missing_counts > 0])    # only show columns WITH missing values
print()

# Percentage missing — more informative than raw counts
print('Missing percentage per column:')
pct = (df_missing.isnull().sum() / len(df_missing) * 100).round(1)
print(pct[pct > 0])

In [ ]:
# ── Strategy 1: Drop rows with any missing value ──────────────────────────────
# Appropriate when: few missing rows AND you can afford to lose them
df_dropped = df_missing.dropna()   # drops rows with NaN in ANY column
print(f'Rows before dropna: {len(df_missing)}')
print(f'Rows after dropna:  {len(df_dropped)}')
print(f'Rows lost: {len(df_missing) - len(df_dropped)}')
print()

# Drop only if specific columns are missing (more targeted)
df_dropped_price = df_missing.dropna(subset=['price'])
print(f'Rows after dropping only missing price: {len(df_dropped_price)}')
print()

# ── Strategy 2: Fill with median ──────────────────────────────────────────────
# Use MEDIAN (not mean) for skewed distributions like price — it is more robust
# to extreme outliers
df_filled = df_missing.copy()
price_median = df_filled['price'].median()
sqft_median  = df_filled['sqft'].median()

df_filled['price'] = df_filled['price'].fillna(price_median)
df_filled['sqft']  = df_filled['sqft'].fillna(sqft_median)

print(f'Price median used for fill: ${price_median:,.0f}')
print(f'Missing values after fill:  {df_filled.isnull().sum().sum()}')
print()

# ── Strategy 3: Forward-fill (good for time-series) ───────────────────────────
# Propagates the LAST known value forward — useful for daily price data
# (not ideal here, shown for completeness)
df_ffill = df_missing.copy()
df_ffill = df_ffill.sort_values('sold_date').fillna(method='ffill')
print('Forward-fill strategy: remaining NaNs =', df_ffill.isnull().sum().sum())

# ── We proceed with the filled DataFrame ─────────────────────────────────────
df = df_filled.copy()
print('\nProceeding with median-filled DataFrame.')

## 9. The `apply` Function — Custom Row/Column Operations

### Plain English
Sometimes you need to do something more complex than simple arithmetic — like categorising a price into a bucket using business logic. `apply()` lets you run **any Python function** on every element or every row.

### Important warning: apply is slow
`apply` runs Python-level loops internally and is **10–100× slower** than vectorised pandas/numpy operations. Only use it when a vectorised approach is not obvious. Prefer `np.where`, `pd.cut`, or column arithmetic when possible.

In [ ]:
# ── apply on a Series (column) — price classification ────────────────────────
# Classify each house by price into four market segments
# Using pandas quantiles ensures equal-sized buckets regardless of price range

q33 = df['price'].quantile(0.33)   # 33rd percentile  ≈ lower-third of market
q66 = df['price'].quantile(0.66)   # 66th percentile  ≈ upper-third of market
q90 = df['price'].quantile(0.90)   # 90th percentile  ≈ top 10%
print(f'Price quantiles → 33%: ${q33:,.0f}  |  66%: ${q66:,.0f}  |  90%: ${q90:,.0f}')

def classify_price(price):
    """Classify a single price value into a market segment."""
    if price < q33:
        return 'budget'
    elif price < q66:
        return 'standard'
    elif price < q90:
        return 'luxury'
    else:
        return 'premium'

# apply() calls classify_price once for every value in the price column
df['price_category'] = df['price'].apply(classify_price)

# Equivalent lambda version (identical result, less readable for complex logic)
# df['price_category'] = df['price'].apply(
#     lambda x: 'budget' if x < q33 else ('standard' if x < q66 else ('luxury' if x < q90 else 'premium'))
# )

print('\nPrice category distribution:')
print(df['price_category'].value_counts())

In [ ]:
# ── apply on rows (axis=1) — combine multiple columns ────────────────────────
# Sometimes the logic needs several columns from the SAME row.
# axis=1 passes each row as a Series to the function.

def value_score(row):
    """
    Compute a simple value score: high sqft, low price, with garage = great deal.
    This is intentionally simple to illustrate row-wise apply.
    """
    sqft_score  = row['sqft'] / 1000       # normalise sqft (larger = better)
    price_score = 500_000 / row['price']   # invert price (cheaper = better)
    garage_bonus = 0.1 if row['garage'] == 1 else 0
    return round(sqft_score + price_score + garage_bonus, 3)

df['value_score'] = df.apply(value_score, axis=1)

# Show the top-5 best-value houses
print('Top 5 best-value houses:')
print(
    df.nlargest(5, 'value_score')
      [['house_id', 'neighborhood', 'sqft', 'price', 'garage', 'value_score']]
)

# ── Vectorised equivalent of the same logic (much faster for large datasets) ──
df['value_score_fast'] = (
    df['sqft'] / 1000
    + 500_000 / df['price']
    + 0.1 * df['garage']
).round(3)

print('\nBoth methods agree:', np.allclose(df['value_score'], df['value_score_fast']))

## 10. `groupby` — Split → Apply → Combine

### Analogy
Imagine sorting your 200 house index cards into four piles — one per neighbourhood. Then you calculate the average price for each pile separately. Finally you lay out all four results side by side. That is exactly what `groupby` does:

1. **Split** the DataFrame by unique values of the grouping column
2. **Apply** an aggregation function to each group
3. **Combine** results into a new (smaller) DataFrame

This pattern is identical to `GROUP BY` in SQL.

In [ ]:
# ── Basic groupby: mean price per neighbourhood ───────────────────────────────
# Result is a Series with neighbourhood names as the index
avg_price_by_hood = df.groupby('neighborhood')['price'].mean()
print('Average price per neighbourhood:')
print(avg_price_by_hood.sort_values(ascending=False).apply(lambda x: f'${x:,.0f}'))
print()

# ── Multiple aggregations in one call ─────────────────────────────────────────
# .agg() accepts a dict of {column: [list of functions]}
price_stats = df.groupby('neighborhood').agg(
    price_mean  = ('price', 'mean'),
    price_median= ('price', 'median'),
    price_std   = ('price', 'std'),
    count       = ('price', 'count')
).round(0)
print('Price statistics per neighbourhood:')
print(price_stats)
print()

# ── Groupby on multiple columns ───────────────────────────────────────────────
# E.g., average price per neighbourhood AND number of bedrooms
price_by_hood_beds = (
    df.groupby(['neighborhood', 'bedrooms'])['price']
      .mean()
      .unstack(level='bedrooms')    # pivot bedrooms to columns for readability
      .round(0)
)
print('Mean price by neighbourhood × bedrooms:')
print(price_by_hood_beds)

In [ ]:
# ── groupby + transform: adding group statistics without collapsing ────────────
# CRITICAL DISTINCTION:
#   groupby().mean()       → returns a SHORT Series (one value per group)
#   groupby().transform()  → returns a LONG Series (same length as original df)
#                            each row gets its GROUP's statistic
#
# Use case: compare each house to its neighbourhood average

# Add neighbourhood mean price as a column on every row
df['hood_mean_price'] = df.groupby('neighborhood')['price'].transform('mean')

# Now compute how much each house deviates from its neighbourhood average
df['price_vs_hood_mean'] = df['price'] - df['hood_mean_price']

print('Sample of houses showing price deviation from neighbourhood mean:')
print(
    df[['house_id', 'neighborhood', 'price', 'hood_mean_price', 'price_vs_hood_mean']]
      .head(8)
      .to_string(index=False)
)
print()

# This is a powerful feature for ML: it encodes LOCAL market information
# A house that is $50K ABOVE its neighbourhood mean is likely high quality
print('Houses furthest above their neighbourhood average (potential overpriced):')
print(
    df.nlargest(5, 'price_vs_hood_mean')
      [['house_id', 'neighborhood', 'price', 'hood_mean_price', 'price_vs_hood_mean']]
)

## 11. `merge` — Combining Two DataFrames

### Analogy
You have two Excel sheets:
- **Sheet 1:** houses with a `neighborhood` column
- **Sheet 2:** neighbourhood attributes (school rating, crime index)

You want to look up each house's neighbourhood rating — that is exactly VLOOKUP in Excel, and exactly `pd.merge()` in pandas.

### Join types (Venn diagram logic)

```
   df1       df2
  ┌───┐     ┌───┐
  │   │ ∩   │   │   INNER  — only rows with a match in BOTH tables
  │   └─────┘   │   LEFT   — all rows from df1, NaN if no match in df2
  │             │   RIGHT  — all rows from df2, NaN if no match in df1
  └─────────────┘   OUTER  — all rows from BOTH (NaN where no match)
```

**Tip:** Always start with a LEFT join when enriching your main dataset — you keep all original rows and just add new columns where available.

In [ ]:
# ── Create a second DataFrame with neighbourhood metadata ─────────────────────
# This simulates a separate data source (e.g., a government school-ranking CSV)
df_schools = pd.DataFrame({
    'neighborhood':  ['Greenwood', 'Downtown', 'Riverside', 'Suburbia'],
    'school_rating': [8.5, 7.2, 6.8, 7.9],     # out of 10
    'crime_index':   [12, 35, 22, 18],           # lower is safer
    'park_count':    [5, 2, 8, 3],               # number of public parks
})
print('Neighbourhood attributes table:')
print(df_schools)
print()

# ── LEFT join: add school / safety data to each house row ─────────────────────
# how='left'  → keep ALL rows from df (left table)
# on='neighborhood' → the shared key column
df_enriched = pd.merge(
    df,           # left table  (200 houses)
    df_schools,   # right table (4 neighbourhoods)
    on='neighborhood',
    how='left'
)

print(f'Rows before merge: {len(df)}')
print(f'Rows after merge:  {len(df_enriched)}')   # still 200 — left join preserves all rows
print(f'Columns after merge: {df_enriched.columns.tolist()}')
print()
print(df_enriched[['house_id', 'neighborhood', 'price', 'school_rating', 'crime_index']].head(6))

In [ ]:
# ── Demonstrate INNER join vs LEFT join difference ────────────────────────────
# Introduce a neighbourhood that has no school data to make the difference visible
df_new_hood = pd.DataFrame({
    'house_id':     ['H201', 'H202'],
    'neighborhood': ['NewTown', 'NewTown'],   # NOT in df_schools
    'price':        [250_000, 310_000],
    'sqft':         [1400, 1700],
})

# Combine into one DataFrame for testing
df_test = pd.concat([df[['house_id', 'neighborhood', 'price', 'sqft']], df_new_hood],
                    ignore_index=True)

left_result  = pd.merge(df_test, df_schools, on='neighborhood', how='left')
inner_result = pd.merge(df_test, df_schools, on='neighborhood', how='inner')

print(f'LEFT  join rows: {len(left_result)}   (keeps NewTown rows with NaN school data)')
print(f'INNER join rows: {len(inner_result)}  (drops NewTown rows — no match in right table)')
print()
print('LEFT join — NewTown rows have NaN for school columns:')
print(left_result[left_result['neighborhood'] == 'NewTown'][['house_id', 'neighborhood', 'school_rating']])

## 12. Visualising the Dataset (Quick Pandas Plots)

Pandas has a built-in `.plot()` method that wraps matplotlib. These are quick-and-dirty plots for exploration — use Matplotlib/Seaborn (Notebook 10) for publication-quality charts.

In [ ]:
# ── Visualise groupby results ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: average price per neighbourhood
avg_by_hood = df_enriched.groupby('neighborhood')['price'].mean().sort_values(ascending=False)
avg_by_hood.plot(
    kind='bar',
    ax=axes[0],
    color=sns.color_palette('husl', 4),
    edgecolor='black',
    linewidth=0.5
)
axes[0].set_title('Average House Price by Neighbourhood', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Neighbourhood')
axes[0].set_ylabel('Average Price ($)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[0].tick_params(axis='x', rotation=15)

# Scatter: school rating vs average price per neighbourhood
hood_stats = df_enriched.groupby('neighborhood').agg(
    avg_price    = ('price', 'mean'),
    school_rating= ('school_rating', 'first')
).reset_index()

axes[1].scatter(hood_stats['school_rating'], hood_stats['avg_price'],
                s=200, color='steelblue', edgecolors='black', zorder=3)
for _, row in hood_stats.iterrows():
    axes[1].annotate(row['neighborhood'],
                     (row['school_rating'], row['avg_price']),
                     textcoords='offset points', xytext=(8, 4), fontsize=9)
axes[1].set_title('School Rating vs Average Price', fontsize=13, fontweight='bold')
axes[1].set_xlabel('School Rating (out of 10)')
axes[1].set_ylabel('Average Price ($)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.suptitle('Pandas groupby + merge in action', y=1.02, fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# ── Price distribution by price category ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count of houses per price category
cat_counts = df['price_category'].value_counts().reindex(['budget', 'standard', 'luxury', 'premium'])
cat_counts.plot(
    kind='bar',
    ax=axes[0],
    color=['#2ecc71', '#3498db', '#9b59b6', '#e74c3c'],
    edgecolor='black',
    linewidth=0.5
)
axes[0].set_title('House Count by Price Category', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Price Category')
axes[0].set_ylabel('Number of Houses')
axes[0].tick_params(axis='x', rotation=0)

# Boxplot: price by age_category
age_order = ['new', 'mid', 'old']
df_age_clean = df[df['age_category'].notna()]
age_groups = [df_age_clean[df_age_clean['age_category'] == cat]['price'].values
              for cat in age_order]
bp = axes[1].boxplot(age_groups, labels=age_order, patch_artist=True)
colors = ['#27ae60', '#e67e22', '#c0392b']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Price Distribution by House Age Category', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Age Category')
axes[1].set_ylabel('Price ($)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.show()

## 13. Summary — The Pandas ML Workflow

Here is a typical pandas data-preparation workflow before feeding data to an ML model:

```python
# Step 1: Load
df = pd.read_csv('houses.csv')

# Step 2: Explore
df.info(); df.describe(); df.isnull().sum()

# Step 3: Clean
df['price'].fillna(df['price'].median(), inplace=True)
df = df.dropna(subset=['sqft'])

# Step 4: Enrich
df = pd.merge(df, neighbourhood_data, on='neighborhood', how='left')

# Step 5: Engineer features
df['price_per_sqft'] = df['price'] / df['sqft']
df['hood_mean_price'] = df.groupby('neighborhood')['price'].transform('mean')
df['price_category']  = pd.cut(df['price'], bins=4, labels=['budget','standard','luxury','premium'])

# Step 6: Split features and target
X = df.drop(columns=['price', 'house_id', 'sold_date'])
y = df['price']

# Step 7: Hand off to ML model
from sklearn.ensemble import RandomForestRegressor
model.fit(X, y)
```

Every step in this workflow uses the techniques you practised in this notebook.

## 14. Self-Check Questions

Try to answer each question **before** expanding the answer.

---

### Q1. What is the difference between `.loc` and `.iloc`?

<details>
<summary>Click to reveal answer</summary>

`.loc` uses **label-based** access — you refer to rows and columns by their names/labels.
`df.loc[5, 'price']` → row with **label** 5, column named `'price'`.

`.iloc` uses **integer position-based** access — pure 0-based numeric positions.
`df.iloc[5, 7]` → the 6th row (0-indexed), 8th column (0-indexed).

The key difference becomes apparent after **sorting or filtering**: if you sort by price and then use `.loc[0]`, you get the row whose *label* is 0 (which may no longer be the first row). `.iloc[0]` always returns the *physically first* row after sorting.

**House dataset example:**
```python
df.loc[10, 'price']      # price of the house with label (index) 10
df.iloc[10, 7]           # price of the 11th row, 8th column
# After df = df.sort_values('price'), these return DIFFERENT rows!
```
</details>

---

### Q2. What is the difference in output shape between `.mean()` and `.transform('mean')` after `groupby`?

<details>
<summary>Click to reveal answer</summary>

```python
df.groupby('neighborhood')['price'].mean()              # → Series of 4 values (one per neighbourhood)
df.groupby('neighborhood')['price'].transform('mean')   # → Series of 200 values (same length as df)
```

- `.mean()` **collapses** the DataFrame — useful for summaries and reports.
- `.transform('mean')` **broadcasts** the group mean back to every original row — useful for feature engineering (each house knows its neighbourhood's average price).
</details>

---

### Q3. You have 200 houses; 15 are missing price data. Should you drop or fill?

<details>
<summary>Click to reveal answer</summary>

**Factors to consider:**

| Factor | Lean toward DROP | Lean toward FILL |
|--------|-----------------|------------------|
| % missing | < 5% | 5–20% |
| Is price the TARGET variable? | Drop (you cannot impute a target) | N/A — never impute the target |
| Is price a FEATURE? | Drop if few rows | Fill with median/neighbourhood mean |
| Is missingness random? | Drop (safe) | Fill if MAR or MNAR |

15/200 = **7.5%** — on the boundary. Since `price` is typically the **target variable** in a house price prediction model, you **cannot impute it** (that would be data leakage). You should **drop** those 15 rows, leaving 185 training examples. If price were a feature, filling with the neighbourhood median would be a reasonable strategy.
</details>

---

### Q4. Write a one-liner to select all houses with more than 3 bedrooms AND price less than $400K.

<details>
<summary>Click to reveal answer</summary>

```python
result = df[(df['bedrooms'] > 3) & (df['price'] < 400_000)]
```

Key rules:
- Use `&` (bitwise AND), **not** Python's `and` keyword
- Each condition **must** be in its own parentheses
- Use `|` for OR, `~` for NOT
</details>

In [ ]:
# ── Final state of the enriched DataFrame ────────────────────────────────────
print('Final enriched DataFrame columns:')
print(df_enriched.columns.tolist())
print()
print('Shape:', df_enriched.shape)
print()
print('Sample rows:')
df_enriched[[
    'house_id', 'neighborhood', 'sqft', 'bedrooms',
    'price', 'price_per_sqft', 'age_category',
    'price_category', 'school_rating'
]].head(8)